In [1]:
import os
from pathlib import Path
import shutil

TEXBIN = "/Library/TeX/texbin"
if Path(TEXBIN).is_dir():
    os.environ["PATH"] = TEXBIN + ":" + os.environ.get("PATH", "")

print("pdflatex now:", shutil.which("pdflatex"))
print("latexmk now:", shutil.which("latexmk"))

pdflatex now: /Library/TeX/texbin/pdflatex
latexmk now: /Library/TeX/texbin/latexmk


### User input & data generation

In [2]:
from openai import OpenAI
import numpy as np, pandas as pd, re, json, os
from pathlib import Path
import hashlib
import subprocess, tempfile, shutil, pathlib
from dotenv import load_dotenv
import re

load_dotenv()
experiment = {
    "context": (
        "Write an observations-style report of phosphate uptake kinetics in saline soils "
        "using time-series batch tests. Each record contains time t (min), the run-specific "
        "parameters Qe1 (mg/g) and k1 (1/min), and the observed qt (mg/g) at time t. "
        "Describe what was done and what was recorded, not how data were generated."
    ),
    "motivation": (
        "Provide a clear lab-note summary: state N, recorded ranges and summary statistics for t, qt, Qe1, and k1, "
        "and describe how qt changes with time across runs with different parameter values. Keep it practical."
    ),

    "NUM_DATASETS": 3,
    "N_OBS": 12,

    # Pseudo-first-order kinetics: only the variables in the relation
    "inputs": {
        "t_(min)"    : ("min",   "uniform", [0.0, 120.0]),
        "Qe1_(mg/g)" : ("mg/g",  "uniform", [0.30, 2.50]),
        "k1_(1/min)" : ("1/min", "uniform", [0.002, 0.080]),
    },

    # qt(t) = Qe1 * (1 - exp(-k1 * t))  (noise only on qt measurement)
    "outputs": {
        "qt_(mg/g)": (
            "mg/g",
            "Qe1_(mg/g) * (1.0 - np.exp(-k1_(1/min) * t_(min)))",
            ("gaussian", [0.0, 0.02])
        ),
    },

    "paper_words": 500,
    "wrap_words": 20,
    "model_id": "gpt-4.1",
    "tok_paper": 1600,
    "tok_titleabs": 500,
    "out_dir": "Output/PseudoFirstOrder_run"
}

experiment["constraints"] = (
    "[INSTRUCTIONS—DO NOT INCLUDE THIS PARAGRAPH IN THE PAPER]\n"
    "Audience: lab collaborators reviewing recorded observations.\n"
    "Style: observational narrative. Use verbs like recorded, observed, measured. "
    "Never use sampled, drawn, generated, simulated, synthetic, ground-truth, noise, template, prompt.\n"
    "Variables allowed anywhere in the paper: t (min), qt (mg/g), Qe1 (mg/g), k1 (1/min). "
    "Do not introduce other variables.\n"
    "Do not print any equations or function forms; refer to the model only in words as a standard pseudo first-order description.\n"
    "Do not include any itemized/bullet lists of variables. Present information as short paragraphs and a single data table.\n"
    f"Use the actual sample size N={experiment['N_OBS']}. Use siunitx units (\\si{{min}}, \\si{{mg\\per g}}, \\si{{1\\per min}}). "
    "Percent must be written as '\\%'.\n"
    "Section headings must be exactly: \\section{Introduction}, \\section{Methods}, \\section{Results}, \\section{Discussion}.\n"
)
experiment["include_variable_bullets"] = False

client = OpenAI()
rng    = np.random.default_rng(2025)
OUT    = Path(experiment["out_dir"]); OUT.mkdir(exist_ok=True, parents=True)

dists  = {
    "uniform": lambda a,b,n: rng.uniform(a,b,n),
    "normal" : lambda m,s,n: rng.normal(m,s,n),
    "int"    : lambda a,b,n: rng.integers(a,b+1,n),
}

safe_pat = re.compile(r'[^0-9a-zA-Z_]')
def safe(col): return safe_pat.sub('_', col)

BAN_PATTERNS = re.compile(
    r"(constraints of this platform|not feasible|token limit|as an ai|i cannot|i can't|"
    r"model cannot|chatgpt|this interface|outline only)",
    re.IGNORECASE
)
def strip_meta(text: str) -> str:
    return "\n".join(ln for ln in text.splitlines() if not BAN_PATTERNS.search(ln)).strip()

def sample_inputs(n):
    return pd.DataFrame({
        col: dists[dist](*params, n)
        for col, (_, dist, params) in experiment["inputs"].items()
    })

def add_outputs(df):
    for name, (_, formula, noise) in experiment["outputs"].items():
        expr = formula.replace('^','**')
        env  = {"np": np}
        for orig in df.columns:
            token     = safe(orig)
            env[token] = df[orig]
            expr      = expr.replace(orig, token)
        y = eval(expr, env)
        if noise[0] == "gaussian":
            mu, sd = noise[1]
            y += rng.normal(mu, sd, len(df))
        df[name] = y
    return df.round(6)

def stats_json(df):
    return json.dumps(
        df.describe().T[["mean","std","min","max"]].round(4).to_dict(), indent=2
    )

### LaTeX formatting helpers

In [3]:
def markdown_emphasis_to_tex(text: str) -> str:
    text = re.sub(r"\*\*(.+?)\*\*", r"\\textbf{\1}", text, flags=re.S)
    text = re.sub(r"(?<!\\)\*(?!\s)(.+?)(?<!\s)\*", r"\\emph{\1}", text, flags=re.S)
    return text

def latex_escape(s: str) -> str:
    if s is None: return ""
    repl = {'\\': r'\textbackslash{}','{': r'\{','}': r'\}','\$': r'\$','&': r'\&',
            '%': r'\%','#': r'\#','_': r'\_','^': r'\^{}','~': r'\~{}'}
    return "".join(repl.get(ch, ch) for ch in s)

_unit_caret_pat   = re.compile(r'([A-Za-z])(\d+)')
_unit_negexp_pat  = re.compile(r'([A-Za-z])-(\d+)')
def _unit_to_si(u: str) -> str:
    u = u.strip()
    u = _unit_caret_pat.sub(r'\1^\2', u)
    u = _unit_negexp_pat.sub(r'\1^{-\2}', u)
    u = u.replace("/", r"\per ")
    return r"\si{" + u + "}"

def _format_header_with_units(colname: str) -> str:
    pretty = colname.replace("_", " ")
    m = re.match(r"^(.*)\((.*)\)\s*$", pretty)
    if m:
        return f"{latex_escape(m.group(1).strip())} ({_unit_to_si(m.group(2).strip())})"
    return latex_escape(pretty)

def df_to_latex_table(df: pd.DataFrame, caption="Generated Data", label="tab:data", floatfmt=6) -> str:
    df2 = df.copy()
    for c in df2.select_dtypes(include="float"):
        df2[c] = df2[c].round(floatfmt)
    cols_fmt = [_format_header_with_units(c) for c in df2.columns]
    col_spec = "c" * len(cols_fmt)
    header = " & ".join(cols_fmt) + r" \\ \midrule"
    rows = "\n".join(" & ".join(map(str, r)) + r" \\" for r in df2.values)
    return (
        "\\begin{table}[h]\n"
        "\\centering\n"
        f"\\caption{{{caption}}}\n"
        f"\\label{{{label}}}\n"
        "\\small\n"
        f"\\begin{{tabular}}{{{col_spec}}}\n"
        "\\toprule\n"
        f"{header}\n"
        f"{rows}\n"
        "\\bottomrule\n"
        "\\end{tabular}\n"
        "\\end{table}\n"
    )

def _topic_phrase_from_context(ctx: str) -> str:
    if not ctx:
        return "the study topic"
    s = ctx.strip()
    s = re.split(r'[.!?\n]', s, 1)[0]
    s = re.sub(r",?\s*so\s+(?:I|we)\b.*$", "", s, flags=re.I)
    s = re.sub(r"^\s*(?:I|we)\s+(?:am|are)\s+", "", s, flags=re.I)
    s = re.sub(r"^\s*(?:this\s+(?:paper|study)\s+)?(?:aims?|seeks?|tries?)\s+to\s+", "", s, flags=re.I)
    s = re.sub(r"^estimating\b",   "estimation of",  s, flags=re.I)
    s = re.sub(r"^predicting\b",   "prediction of",  s, flags=re.I)
    s = re.sub(r"^measuring\b",    "measurement of", s, flags=re.I)
    s = re.sub(r"^analy[sz]ing\b", "analysis of",    s, flags=re.I)
    s = re.sub(r"^investigating\b","investigation of", s, flags=re.I)
    s = s.strip(" ,")
    return latex_escape(s or "the study topic")


def inject_table_into_results_tex(paper: str, table_tex: str) -> str:
    # Figure out label and a compact preview for the lead sentence
    mlabel = re.search(r"\\label\{([^}]+)\}", table_tex)
    label  = mlabel.group(1) if mlabel else "tab:data"

    between = re.search(r"\\midrule(.*?)\\bottomrule", table_tex, re.S)
    row_count = 0
    if between:
        row_count = len([ln for ln in between.group(1).splitlines() if "\\" in ln])

    mhdr = re.search(r"\\toprule\s*(.*?)\\\\\s*\\midrule", table_tex, re.S)
    header_cols = []
    if mhdr:
        header_cols = [re.sub(r"\s+", " ", h.strip()) for h in mhdr.group(1).split("&")]

    topic = _topic_phrase_from_context(experiment.get("context",""))
    PREVIEW_K = 6
    if len(header_cols) <= PREVIEW_K:
        cols_preview, tail = ", ".join(header_cols), ""
    else:
        cols_preview = ", ".join(header_cols[:PREVIEW_K])
        tail = f" (+{len(header_cols)-PREVIEW_K} more)"

    # Lead sentence (observational, no distributions, no formulas)
    templates = [
        r"Table~\ref{%LABEL%} (N=%N%) presents recorded observations for %TOPIC% with columns: %COLS%%TAIL%.",
        r"For %TOPIC%, Table~\ref{%LABEL%} reports N=%N% recorded observations across the columns: %COLS%%TAIL%.",
        r"The recorded dataset in Table~\ref{%LABEL%} (N=%N%) lists %COLS%%TAIL% for %TOPIC%.",
    ]
    t_idx = (len(header_cols) + row_count + sum(map(ord, label))) % len(templates)
    lead_sentence = (templates[t_idx]
                     .replace("%LABEL%", label)
                     .replace("%N%", str(row_count))
                     .replace("%TOPIC%", topic)
                     .replace("%COLS%", cols_preview)
                     .replace("%TAIL%", tail))

    # No “Inputs are drawn…”, no itemize bullets
    lead = "\n\n" + lead_sentence + "\n\n"

    # Insert at the start of Results section; otherwise append to end
    pattern = re.compile(r'^[ \t]*#{1,6}\s*Results\b.*$', re.IGNORECASE | re.MULTILINE)
    mr = pattern.search(paper)
    block = lead + table_tex + "\n"
    if mr:
        insert_at = mr.end()
        return paper[:insert_at] + block + paper[insert_at:]
    else:
        return paper + "\n\n" + block


def md_to_tex(text: str) -> str:
    text = markdown_emphasis_to_tex(text)
    out = []
    for ln in text.splitlines():
        if ln.startswith("##### "): out.append(r"\paragraph{"      + latex_escape(ln[6:]) + "}")
        elif ln.startswith("#### "): out.append(r"\subsubsection{" + latex_escape(ln[5:]) + "}")
        elif ln.startswith("### "):  out.append(r"\subsubsection{" + latex_escape(ln[4:]) + "}")
        elif ln.startswith("## "):   out.append(r"\subsection{"    + latex_escape(ln[3:]) + "}")
        elif ln.startswith("# "):    out.append(r"\section{"       + latex_escape(ln[2:]) + "}")
        else:                        out.append(ln)
    return "\n".join(out)

def clean_title_for_latex(s: str) -> str:
    if not s: return ""
    s = re.sub(r"\$\$.*?\$\$|\$[^$]*\$|\\\[.*?\\\]|\\\((?:.|\n)*?\\\)", "", s, flags=re.S)
    for cmd in ["textbf","emph","textit","textsc","texttt","textsf","mathrm"]:
        s = re.sub(rf"\\{cmd}\{{(.*?)\}}", r"\1", s, flags=re.S)
    s = re.sub(r"\\[a-zA-Z]+\*?(?:\[[^\]]*\])?\{(.*?)\}", r"\1", s, flags=re.S)
    s = re.sub(r"\\[a-zA-Z]+\*?(?:\[[^\]]*\])?", "", s)
    s = re.sub(r"\*\*(.+?)\*\*", r"\1", s, flags=re.S)
    s = re.sub(r"(?<!\\)\*(?!\s)(.+?)(?<!\s)\*", r"\1", s, flags=re.S)
    s = re.sub(r"[ \t]+", " ", s).strip()
    return s

def build_tex_doc(title: str, abstract: str, body_tex: str) -> str:
    safe_title = latex_escape(clean_title_for_latex(title))
    return (
        "\\documentclass[11pt]{article}\n"
        "\\usepackage[a4paper,margin=1in]{geometry}\n"
        "\\usepackage{booktabs}\n"
        "\\usepackage{amsmath, amssymb}\n"
        "\\usepackage{siunitx}\n"
        "\\usepackage{hyperref}\n"
        "\\usepackage{caption}\n"
        "\\captionsetup[table]{position=top}\n\n"
        "\\title{" + safe_title + "}\n"
        "\\author{}\n"
        "\\date{}\n\n"
        "\\begin{document}\n"
        "\\maketitle\n\n"
        "\\begin{abstract}\n"
        + markdown_emphasis_to_tex(strip_meta(abstract)) + "\n"
        "\\end{abstract}\n\n"
        + body_tex + "\n\n"
        "\\end{document}\n"
    )

def _json_fix_backslashes(s: str) -> str:
    return re.sub(r'\\(?![\\/"bfnrtu])', r'\\\\', s)

def strip_latex_math(text: str) -> str:
    patterns = [
        r"\$\$.*?\$\$",
        r"\$[^$]*\$",
        r"\\\[.*?\\\]",
        r"\\\((?:.|\n)*?\\\)",
        r"\\begin\{equation\*?\}.*?\\end\{equation\*?\}",
        r"\\begin\{align\*?\}.*?\\end\{align\*?\}",
        r"\\begin\{gather\*?\}.*?\\end\{gather\*?\}",
    ]
    for pat in patterns:
        text = re.sub(pat, "", text, flags=re.S)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text

def fix_malformed_environments(tex: str) -> str:
    def _clean_env(match, is_begin=True):
        env = match.group(1)
        env = re.split(r'\\', env, 1)[0]
        env = re.sub(r'[^A-Za-z*]', '', env)
        tag = "begin" if is_begin else "end"
        return fr"\{tag}{{{env}}}"
    tex = re.sub(r"\\begin\{([^}]*)\}",  lambda m: _clean_env(m, True),  tex)
    tex = re.sub(r"\\end\{([^}]*)\}",    lambda m: _clean_env(m, False), tex)
    stack, out = [], []
    token_pat = re.compile(r"\\(begin|end)\{([A-Za-z*]+)\}")
    idx = 0
    for m in token_pat.finditer(tex):
        out.append(tex[idx:m.end()])
        idx = m.end()
        typ, env = m.group(1), m.group(2)
        if typ == "begin":
            stack.append(env)
        else:
            if stack and stack[-1] == env:
                stack.pop()
            else:
                out[-1] = out[-1].replace(m.group(0), "")
    out.append(tex[idx:])
    for env in reversed(stack):
        out.append(f"\n\\end{{{env}}}\n")
    return "".join(out)

### Latex complier for the verification of the generated Latex code

In [4]:
import shutil, subprocess, tempfile, pathlib, os

def _tex_engine_cmd():
    # prefer latexmk; fall back to common engines
    for cmd in (
        ["latexmk", "-pdf", "-interaction=nonstopmode", "-halt-on-error"],
        ["pdflatex", "-interaction=nonstopmode", "-halt-on-error"],
        ["lualatex", "-interaction=nonstopmode", "-halt-on-error"],
        ["xelatex", "-interaction=nonstopmode", "-halt-on-error"],
    ):
        if shutil.which(cmd[0]):
            return cmd
    return None


def validate_tex_source(tex_source: str, base_name: str, timeout_sec: int = 120):
    cmd_base = _tex_engine_cmd()
    if not cmd_base:
        return None, "TeX engine not found"

    with tempfile.TemporaryDirectory() as tmpdir:
        tex_path = pathlib.Path(tmpdir) / f"{base_name}.tex"
        tex_path.write_text(tex_source, encoding="utf-8")

        proc = subprocess.run(
            cmd_base + [tex_path.name],
            cwd=tmpdir,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            timeout=timeout_sec)
        ok  = (proc.returncode == 0)
        log = proc.stdout.decode("utf-8", errors="ignore")
        return ok, log


def compile_tex_to_pdf(tex_source: str, base_name: str, out_pdf_path: pathlib.Path, timeout_sec: int = 180):
    """
    Compile LaTeX source to a PDF at out_pdf_path.
    Saves ONLY PDF (not tex) to your OUT dir.
    Returns: (ok: bool, log: str)
    """
    cmd_base = _tex_engine_cmd()
    if not cmd_base:
        return False, "TeX engine not found on PATH. Try ensure_tex_on_path()."

    out_pdf_path = pathlib.Path(out_pdf_path)
    out_pdf_path.parent.mkdir(parents=True, exist_ok=True)

    with tempfile.TemporaryDirectory() as tmpdir:
        tmpdir = pathlib.Path(tmpdir)
        tex_path = tmpdir / f"{base_name}.tex"
        tex_path.write_text(tex_source, encoding="utf-8")

        log_all = []

        if cmd_base[0] == "latexmk":
            # latexmk handles multiple passes automatically
            proc = subprocess.run(
                cmd_base + [tex_path.name],
                cwd=tmpdir,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                timeout=timeout_sec
            )
            log_all.append(proc.stdout.decode("utf-8", errors="ignore"))
            ok = (proc.returncode == 0)
        else:
            # engines often need 2 passes for refs/tables
            ok = True
            for _ in range(2):
                proc = subprocess.run(
                    cmd_base + [tex_path.name],
                    cwd=tmpdir,
                    stdout=subprocess.PIPE,
                    stderr=subprocess.STDOUT,
                    timeout=timeout_sec
                )
                log_all.append(proc.stdout.decode("utf-8", errors="ignore"))
                if proc.returncode != 0:
                    ok = False
                    break

        pdf_path = tmpdir / f"{base_name}.pdf"
        if ok and pdf_path.exists():
            shutil.copy2(pdf_path, out_pdf_path)
            return True, "\n".join(log_all)

        return False, "\n".join(log_all)

### LLM calls & main loop

In [5]:
def llm_paper(stats_json: str) -> str:
    N = experiment["N_OBS"]

    prompt = (
        f"{experiment['context']}\n\n"
        f"Motivation: {experiment['motivation']}\n\n"
        f"{experiment.get('constraints','')}\n\n"
        f"Dataset summary statistics (JSON):\n{stats_json}\n\n"
        f"Write an ~{experiment['paper_words']}-word LaTeX paper with sections:\n"
        "# Introduction\n# Methods\n# Results\n# Discussion\n\n"
        "• Use LaTeX emphasis (\\textbf{}, \\emph{}). Do NOT use Markdown.\n"
        "• Report quantitative observations (ranges, means/SDs, optional 95% CIs) and the tested condition ranges.\n"
        "• Do NOT introduce or display any equations. Do NOT include an Inputs/Outputs bullet list.\n"
        "• Do NOT mention dataset generation, scripts, prompts, ground-truth formulas, or noise.\n"
    )
    res = client.chat.completions.create(
        model=experiment["model_id"],
        messages=[{"role": "user", "content": prompt}],
        max_tokens=experiment["tok_paper"],
        temperature=0.4,
    )
    return strip_meta(res.choices[0].message.content.strip())


def llm_title_abs(paper):
    prompt = (
        "Provide JSON ONLY with keys 'title' and 'abstract'. "
        "Use LaTeX emphasis (\\textbf{}, \\emph{}) not Markdown. "
        "In JSON string values, ESCAPE EVERY BACKSLASH (e.g., write \\\\emph{}, not \\emph{}). "
        "Do NOT mention platform constraints/tokens. "
        "The ABSTRACT MUST BE PROSE ONLY with NO MATH: do not include $...$, $$...$$, \\(\\), \\[\\], "
        "or any equation environments.\n\n"
        "Paper:\n" + paper)
    res  = client.chat.completions.create(
        model=experiment["model_id"],
        messages=[{"role":"user","content":prompt}],
        max_tokens=experiment["tok_titleabs"], temperature=0.4)
    m = re.search(r"\{.*\}", res.choices[0].message.content, re.S)
    if not m:
        raise ValueError("Model did not return JSON for title/abstract.")
    block = m.group(0)
    fixed = _json_fix_backslashes(block)
    try:
        meta = json.loads(fixed)
    except json.JSONDecodeError:
        meta = json.loads(block.replace("\\", "\\\\"))
    abstract = meta.get("abstract","")
    abstract = "\n".join(
        ln for ln in abstract.splitlines()
        if not ln.lower().lstrip().startswith("title:")
    ).strip()
    abstract = strip_meta(abstract)
    abstract = strip_latex_math(abstract)
    abstract = markdown_emphasis_to_tex(abstract)
    meta["abstract"] = abstract
    candidates = meta.get("titles") if isinstance(meta.get("titles"), list) else [meta.get("title","")]
    chosen = choose_diverse_title(candidates, key=paper)
    meta["title"] = clean_title_for_latex(chosen)
    return meta

def _normalize_ws(s: str) -> str:
    return re.sub(r"\s+", " ", s or "").strip()

USED_TITLES = set()

def llm_alternate_titles(paper, banned):
    prompt = (
        "Return JSON ONLY with key 'titles' as an array of 8 distinct candidate titles "
        "(8–14 words each). Titles must be derived from and faithful to the paper content "
        "provided below. Do not introduce topics that are not inferable from this content. "
        "Use plain text (no LaTeX/math). "
        "Avoid any title in this banned list:\n"
        + json.dumps(sorted(list(banned))) +
        "\n\nPaper:\n" + paper
    )
    res = client.chat.completions.create(
        model=experiment['model_id'],
        messages=[{"role": "user", "content": prompt}],
        max_tokens=experiment["tok_titleabs"],
        temperature=0.8, presence_penalty=0.6, frequency_penalty=0.3
    )
    m = re.search(r"\{.*\}", res.choices[0].message.content, re.S)
    if not m:
        return []
    block = _json_fix_backslashes(m.group(0))
    try:
        data = json.loads(block)
    except json.JSONDecodeError:
        data = json.loads(m.group(0).replace("\\", "\\\\"))
    raw = data.get("titles", [])
    seen, out = set(), []
    def _tokens(s: str):
        return re.findall(r"[A-Za-z][A-Za-z0-9\-]{2,}", (s or "").lower())
    stop = {
        "the","and","for","with","from","into","that","this","these","those",
        "study","paper","result","results","method","methods","introduction",
        "discussion","conclusion","data","dataset","analysis","approach",
        "using","use","based","model","models","over","under","between","within",
        "across","about","without","against","toward","towards","through"
    }
    vocab = {t for t in _tokens(paper) if t not in stop}
    for t in raw:
        tt = clean_title_for_latex(_normalize_ws(t))
        if not tt:
            continue
        low = tt.lower()
        if low in seen or tt in banned:
            continue
        tt_tokens = [w for w in _tokens(tt) if w not in stop]
        if not tt_tokens:
            continue
        overlap = sum(1 for w in tt_tokens if w in vocab)
        if overlap >= max(3, len(tt_tokens) // 3):
            seen.add(low)
            out.append(tt)
    return out

def choose_diverse_title(candidates, key: str) -> str:
    uniq, seen = [], set()
    for t in candidates or []:
        tt = _normalize_ws(t)
        if not tt: continue
        low = tt.lower()
        if low in seen: continue
        seen.add(low); uniq.append(tt)
    if not uniq:
        return "Untitled"
    def score(t):
        n = len(t.split())
        return -abs(n - 11)
    uniq.sort(key=score, reverse=True)
    h = int(hashlib.sha256(key.encode("utf-8")).hexdigest(), 16)
    return uniq[h % len(uniq)]


def llm_table_caption(context: str, motivation: str, stats_json: str, cols: list[str]) -> str:
    prompt = (
        "Return JSON ONLY with key 'caption'.\n"
        "Write a single-sentence LaTeX table caption that matches the study context and what the table contains.\n"
        "The caption must describe what was recorded/measured (observational), not how it was produced.\n"
        "Do NOT use words like generated, simulated, synthetic, sampled, drawn, ground-truth, noise.\n"
        "Use \\emph{} or \\textbf{} only if helpful.\n"
        "If N is known, include it as (N = ...).\n"
        "Caption should align with column names and the experiment context.\n\n"
        f"Context:\n{context}\n\n"
        f"Motivation:\n{motivation}\n\n"
        f"Columns:\n{json.dumps(cols)}\n\n"
        f"Summary statistics (JSON):\n{stats_json}\n"
    )
    res = client.chat.completions.create(
        model=experiment["model_id"],
        messages=[{"role": "user", "content": prompt}],
        max_tokens=250,
        temperature=0.4,
    )
    m = re.search(r"\{.*\}", res.choices[0].message.content, re.S)
    if not m:
        return "Summary statistics for recorded measurements."
    block = _json_fix_backslashes(m.group(0))
    try:
        data = json.loads(block)
    except json.JSONDecodeError:
        data = json.loads(m.group(0).replace("\\", "\\\\"))
    cap = (data.get("caption") or "").strip()
    cap = strip_meta(cap)
    cap = strip_latex_math(cap)
    cap = markdown_emphasis_to_tex(cap)
    return cap if cap else "Summary statistics for recorded measurements."

### Main loop

In [6]:
def sanitize_tex(tex: str) -> str:
    # Fix siunitx percent
    tex = tex.replace(r'\si{%}', r'\si{\percent}')
    # Fix doubled backslashes from LLM: \\textbf, \\emph, \\si, \\num, \\qty
    tex = re.sub(r'\\{2}(textbf|emph|si|SI|num|qty)\b', r'\\\1', tex)
    # Nicer header label: "q e" -> "$q_e$" (surrounded by non-word boundaries)
    tex = re.sub(r'(\W)q\s+e(\W)', r'\1$q_e$\2', tex)
    # Escape bare % outside \si{...}
    tex = re.sub(r'(?<!\\)%', r'\\%', tex)
    tex = re.sub( r'The dataset summarized in Table~\\ref\{[^}]+\}.*?\\begin\{table\}', r'\\begin{table}', tex, flags=re.S )
    # 2) Kill any line that contains np.*** or code-like noise in prose
    tex = re.sub(r'^.*\bnp\.(minimum|maximum|clip)\b.*$\n?', '', tex, flags=re.M)
    # (optional) also drop Uniform(…), \mathcal{N}(…) recipe lines if they sneak in
    tex = re.sub(r'^.*\bUniform\(|\\mathcal\{N\}\s*\(.*$\n?', '', tex, flags=re.M)

    # 3) Remove the stray outline/toc block some runs append at the very end
    tex = re.sub(r'\n\s*Introduction\s+Methods.*?Conclusion\s*$', '', tex, flags=re.S)
    # collapse erroneous linebreak-before-percent
    tex = tex.replace('\\\\%', r'\%')
    return tex

def normalize_sections(tex: str) -> str:
    # Fix the common broken patterns
    tex = tex.replace(r'\section\emph{{', r'\section{')
    tex = tex.replace(r'\section}{',    r'\section{')
    tex = tex.replace(r'\section\emph{', r'\section{')
    tex = tex.replace(r'\section{{',     r'\section{')

    # Canonicalize any \section line that contains one of the standard titles
    titles = ["Introduction", "Methods", "Results", "Discussion"]
    for t in titles:
        tex = re.sub(
            rf'(?m)^\s*\\section[^\n]*{re.escape(t)}[^\n]*$',
            rf'\\section{{{t}}}',
            tex
        )
    return tex

In [7]:
MAX_RETRIES = 2   # extra attempts after the first failure

for i in range(1, experiment["NUM_DATASETS"] + 1):
    print(f"▶ Dataset {i}/{experiment['NUM_DATASETS']}")

    df        = add_outputs(sample_inputs(experiment["N_OBS"]))
    stats     = stats_json(df)

    attempt   = 0
    while True:
        attempt += 1
        paper_md = llm_paper(stats)
        label    = f"tab:data{i:02d}"
        cap = llm_table_caption( experiment["context"], experiment["motivation"], stats, list(df.columns) )
        table_tex = df_to_latex_table(df, caption=cap, label=label)
        paper_md_with_table = inject_table_into_results_tex(paper_md, table_tex)
        body_tex = md_to_tex(paper_md_with_table)

        meta      = llm_title_abs(paper_md_with_table)
        alts      = llm_alternate_titles(paper_md_with_table + "\n\nABSTRACT:\n" + meta.get("abstract", ""), USED_TITLES)
        candidates = [t for t in alts if t and t not in USED_TITLES] or \
                     (meta.get("titles") if isinstance(meta.get("titles"), list) else [meta.get("title", "")])
        title      = choose_diverse_title(candidates, key=paper_md_with_table)
        if not title or title in USED_TITLES:
            base  = clean_title_for_latex(meta.get("title", "")) or "Untitled"
            title = base if base not in USED_TITLES else f"{base} — Study {i:02d}"
        USED_TITLES.add(title)
        meta["title"] = title

        tex_doc = build_tex_doc(meta["title"], meta["abstract"], body_tex)
        tex_doc = sanitize_tex(tex_doc) 
        tex_doc = normalize_sections(tex_doc) 
        pdf_path = OUT / f"paper{i:02d}.pdf"
        ok, log = compile_tex_to_pdf(tex_doc, f"paper{i:02d}", pdf_path)
        if ok is None:
            (OUT / f"paper{i:02d}.tex").write_text(tex_doc, encoding="utf-8")
            print("   ⚠ validation skipped (no TeX engine); saved", f"paper{i:02d}.tex")
            break

        if ok:
            print("   ✔ compiled & saved", pdf_path.name)
            break

        print(f"   ✖ compile failed (attempt {attempt}/{MAX_RETRIES + 1})")
        if attempt > MAX_RETRIES:
            log_dir = OUT / "logs"; log_dir.mkdir(exist_ok=True)
            (log_dir / f"paper{i:02d}_compile.log").write_text(log, encoding="utf-8")
            print("     saved compile log to", f"logs/paper{i:02d}_compile.log")
            break

print("🎉  Done. Files in:", OUT.resolve())


▶ Dataset 1/3
   ✔ compiled & saved paper01.pdf
▶ Dataset 2/3
   ✔ compiled & saved paper02.pdf
▶ Dataset 3/3
   ✔ compiled & saved paper03.pdf
🎉  Done. Files in: /Users/md_halim_mondol/Desktop/STEPS-LLM/steps-llm/Src/Output/PseudoFirstOrder_run
